# Vectors and Norms

**Goal:** Implement L1, L2, and Lp norms plus cosine similarity from scratch in PyTorch, validate against `torch.linalg.norm` and `torch.nn.functional.cosine_similarity`, and build intuition for when each metric is used.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import torch
import torch.nn.functional as F


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()  # reads config.toml -> device/seed/dtype (mps on Apple Silicon)
print("running on:", device)

running on: mps


## From Scratch: L1, L2, and General Lp Norms

For a vector x in R^d:

| Norm | Formula |
|------|---------|
| L1   | `sum_i |x_i|` |
| L2   | `sqrt(sum_i x_i^2)` |
| Lp   | `(sum_i |x_i|^p)^(1/p)` |
| L-inf | `max_i |x_i|` |

In [2]:
def l1_norm(x: torch.Tensor) -> torch.Tensor:
    """L1 norm: sum of absolute values."""
    return x.abs().sum()


def l2_norm(x: torch.Tensor) -> torch.Tensor:
    """L2 norm: Euclidean length."""
    return (x * x).sum().sqrt()


def lp_norm(x: torch.Tensor, p: float) -> torch.Tensor:
    """General Lp norm.

    Args:
        x: Input vector.
        p: Order of the norm (>= 1).

    Returns:
        Scalar norm value.
    """
    if p == float("inf"):
        return x.abs().max()
    return (x.abs() ** p).sum() ** (1.0 / p)


x = torch.tensor([3.0, -4.0, 0.0, 1.0], device=device)

print("x:", x)
print("L1 (scratch):", l1_norm(x).item())
print("L2 (scratch):", l2_norm(x).item())
print("L3 (scratch):", lp_norm(x, 3).item())
print("Linf (scratch):", lp_norm(x, float("inf")).item())

x: tensor([ 3., -4.,  0.,  1.], device='mps:0')
L1 (scratch): 8.0
L2 (scratch): 5.099019527435303
L3 (scratch): 4.514358043670654
Linf (scratch): 4.0


## Validation Against `torch.linalg.norm`

In [3]:
ref_l1   = torch.linalg.vector_norm(x, ord=1)
ref_l2   = torch.linalg.vector_norm(x, ord=2)
ref_l3   = torch.linalg.vector_norm(x, ord=3)
ref_linf = torch.linalg.vector_norm(x, ord=float("inf"))

assert torch.allclose(l1_norm(x),                  ref_l1,   atol=1e-5), "L1 mismatch"
assert torch.allclose(l2_norm(x),                  ref_l2,   atol=1e-5), "L2 mismatch"
assert torch.allclose(lp_norm(x, 3),               ref_l3,   atol=1e-5), "L3 mismatch"
assert torch.allclose(lp_norm(x, float("inf")),    ref_linf, atol=1e-5), "Linf mismatch"

print("L1  scratch =", l1_norm(x).item(),              "  ref =", ref_l1.item(),   " ✓")
print("L2  scratch =", l2_norm(x).item(),              "  ref =", ref_l2.item(),   " ✓")
print("L3  scratch =", lp_norm(x, 3).item(),           "  ref =", ref_l3.item(),   " ✓")
print("Linf scratch=", lp_norm(x, float("inf")).item(),"  ref =", ref_linf.item(), " ✓")
print("\nAll norms match torch.linalg.vector_norm ✓")

L1  scratch = 8.0   ref = 8.0  ✓
L2  scratch = 5.099019527435303   ref = 5.099019527435303  ✓
L3  scratch = 4.514358043670654   ref = 4.514358043670654  ✓
Linf scratch= 4.0   ref = 4.0  ✓

All norms match torch.linalg.vector_norm ✓


## Idiomatic PyTorch Norm One-Liners

In [4]:
print("L1:", torch.linalg.vector_norm(x, ord=1).item())
print("L2:", torch.linalg.vector_norm(x, ord=2).item())
print("Linf:", torch.linalg.vector_norm(x, ord=float("inf")).item())

# Batched: norm across rows of a matrix
X_batch = torch.randn(5, 8, device=device)
row_norms = torch.linalg.vector_norm(X_batch, ord=2, dim=1)  # (5,) L2 norm per row
print("\nBatch row L2 norms shape:", row_norms.shape)
print("Row norms:", row_norms)

L1: 8.0
L2: 5.099019527435303
Linf: 4.0

Batch row L2 norms shape: torch.Size([5])
Row norms: tensor([2.5375, 2.7641, 2.2877, 2.3532, 2.1565], device='mps:0')


## From Scratch: Cosine Similarity

`cos_sim(x, y) = (x dot y) / (||x||_2 * ||y||_2)`

Cosine similarity removes magnitude — only direction matters.  
Range: [-1, 1]; 1 means identical direction, 0 means orthogonal, -1 means opposite.

In [5]:
def cosine_similarity_scratch(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Cosine similarity between two 1-D vectors.

    Args:
        x: Vector of shape (d,).
        y: Vector of shape (d,).
        eps: Small value for numerical stability.

    Returns:
        Scalar cosine similarity in [-1, 1].
    """
    dot = (x * y).sum()
    norm_x = l2_norm(x).clamp(min=eps)
    norm_y = l2_norm(y).clamp(min=eps)
    return dot / (norm_x * norm_y)


a = torch.tensor([1.0, 0.0, 1.0], device=device)
b = torch.tensor([1.0, 0.0, 0.0], device=device)
c = torch.tensor([0.0, 1.0, 0.0], device=device)  # orthogonal to a

print("cos_sim(a, b):", cosine_similarity_scratch(a, b).item(), " (same direction, partial overlap)")
print("cos_sim(a, c):", cosine_similarity_scratch(a, c).item(), " (orthogonal, should be 0)")
print("cos_sim(a, a):", cosine_similarity_scratch(a, a).item(), " (identical, should be 1)")

cos_sim(a, b): 0.7071067690849304  (same direction, partial overlap)
cos_sim(a, c): 0.0  (orthogonal, should be 0)
cos_sim(a, a): 1.0000001192092896  (identical, should be 1)


## Validation Against `torch.nn.functional.cosine_similarity`

In [6]:
# F.cosine_similarity expects batched input (N, D)
ref_ab = F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0), dim=1)
ref_ac = F.cosine_similarity(a.unsqueeze(0), c.unsqueeze(0), dim=1)
ref_aa = F.cosine_similarity(a.unsqueeze(0), a.unsqueeze(0), dim=1)

assert torch.allclose(cosine_similarity_scratch(a, b), ref_ab.squeeze(), atol=1e-5), "cos(a,b) mismatch"
assert torch.allclose(cosine_similarity_scratch(a, c), ref_ac.squeeze(), atol=1e-5), "cos(a,c) mismatch"
assert torch.allclose(cosine_similarity_scratch(a, a), ref_aa.squeeze(), atol=1e-5), "cos(a,a) mismatch"
print("All cosine similarity values match F.cosine_similarity ✓")

# Larger random vectors
x_rand = torch.randn(128, device=device)
y_rand = torch.randn(128, device=device)
cs_scratch = cosine_similarity_scratch(x_rand, y_rand)
cs_ref     = F.cosine_similarity(x_rand.unsqueeze(0), y_rand.unsqueeze(0), dim=1).squeeze()
assert torch.allclose(cs_scratch, cs_ref, atol=1e-5)
print("128-D random vectors:", cs_scratch.item(), "(scratch)  vs", cs_ref.item(), "(ref)  ✓")

All cosine similarity values match F.cosine_similarity ✓
128-D random vectors: 0.02590002305805683 (scratch)  vs 0.02590002864599228 (ref)  ✓


## Euclidean Distance

`dist_2(x, y) = ||x - y||_2` — accounts for both direction and magnitude.

In [7]:
def euclidean_distance(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """L2 distance between two vectors."""
    return l2_norm(x - y)


print("Euclidean distance a-b:", euclidean_distance(a, b).item())
print("torch.dist check:      ", torch.dist(a, b, p=2).item())
assert torch.allclose(euclidean_distance(a, b), torch.dist(a, b, p=2), atol=1e-5)
print("Euclidean distance matches torch.dist ✓")

Euclidean distance a-b: 1.0


torch.dist check:       1.0
Euclidean distance matches torch.dist ✓


## Takeaways: When to Use Each Norm

| Norm | Use Case |
|------|----------|
| **L1** | Sparsity-inducing regularization (LASSO); encourages zero weights; robust to outliers |
| **L2** | Ridge regularization; Euclidean distance; gradient norms; least-squares objectives |
| **Linf** | Adversarial robustness; worst-case guarantees; checking max perturbation |
| **Cosine** | Text/embedding retrieval where magnitude varies; direction-only similarity |

- Cosine similarity is preferred for embeddings because model outputs may have arbitrary scale.
- L1 penalties produce *sparse* weight vectors; L2 penalties produce *small but non-zero* weights.
- Gradient norms (L2) are the standard diagnostic for training stability (gradient clipping threshold).